# FER-7 Multi-Dataset Training (v4) — Google Colab

This notebook runs the v4 training script on a **free Colab GPU**.

## Setup Steps
1. Upload your data to Google Drive (one-time)
2. Mount Drive in Colab
3. Fix image paths in CSV
4. Train!

### Before running this notebook:
On your local PC, **zip** the `processed_faces/` folder and copy these files to a Google Drive folder called `FER`:
```
Google Drive/
  FER/
    processed_faces.zip          <-- zip of processed_faces/ folder
    unified_fer7_processed_au.csv  <-- from dataset_csv/
```

To zip on Windows (PowerShell):
```powershell
Compress-Archive -Path processed_faces -DestinationPath processed_faces.zip
```

## 1. Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime > Change runtime type > GPU")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check your files are there
import os
FER_DIR = '/content/drive/MyDrive/FER'
print("Files in FER folder:")
for f in os.listdir(FER_DIR):
    size_mb = os.path.getsize(os.path.join(FER_DIR, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

## 3. Unzip processed faces (to Colab local disk for speed)

In [ ]:
import zipfile

ZIP_PATH = os.path.join(FER_DIR, 'processed_faces.zip')
EXTRACT_TO = '/content/data'
os.makedirs(EXTRACT_TO, exist_ok=True)

if not os.path.isdir(os.path.join(EXTRACT_TO, 'processed_faces')):
    print(f"Unzipping {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_TO)
    print("Done!")
else:
    print("Already unzipped.")

# Quick check
for d in os.listdir(os.path.join(EXTRACT_TO, 'processed_faces')):
    print(f"  {d}/")

## 4. Fix image paths in CSV
The CSV has Windows paths like `C:\Users\carla\...`. We replace them with the Colab paths.

In [ ]:
import pandas as pd

CSV_SRC = os.path.join(FER_DIR, 'unified_fer7_processed_au.csv')
CSV_LOCAL = '/content/data/unified_fer7_processed_au.csv'

OLD_PREFIX = r'C:\Users\carla\PyCharmMiscProject'
NEW_PREFIX = '/content/data'

df = pd.read_csv(CSV_SRC, low_memory=False)
print(f"Rows: {len(df)}")
print(f"Sample path BEFORE: {df['image'].iloc[0]}")

# Replace Windows backslash paths with Unix forward-slash paths
df['image'] = (
    df['image']
    .str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    .str.replace('\\', '/', regex=False)
)

# Also fix orig_image if it exists
if 'orig_image' in df.columns:
    df['orig_image'] = (
        df['orig_image']
        .str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
        .str.replace('\\', '/', regex=False)
    )

print(f"Sample path AFTER:  {df['image'].iloc[0]}")

# Verify a few images exist
found = df['image'].head(20).apply(os.path.isfile).sum()
print(f"Verified: {found}/20 images found")

df.to_csv(CSV_LOCAL, index=False)
print(f"Saved fixed CSV to {CSV_LOCAL}")

## 5. Training Script
This is the full `cnn_train_multidataset_fer7_soft_au.py` written as a cell.

In [ ]:
%%writefile /content/cnn_train_multidataset_fer7_soft_au.py
"""
FER-7 multi-dataset training script  - v4 (Colab version)
"""

import os
import json
import argparse
from pathlib import Path
from typing import Dict, Any, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image, ImageFile
from torchvision import transforms, models

from sklearn.metrics import f1_score, classification_report

ImageFile.LOAD_TRUNCATED_IMAGES = True

FER7    = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
LAB2ID  = {c: i for i, c in enumerate(FER7)}
RARE_CLASSES = {"disgust", "fear"}

P_COLS = ["p_angry", "p_disgust", "p_fear", "p_happy", "p_sad", "p_surprise", "p_neutral"]
AU_COLS = [
    "au_eye_ear_l", "au_eye_ear_r",
    "au_eye_open_l", "au_eye_open_r",
    "au_mouth_open", "au_mouth_width",
    "au_brow_raise_l", "au_brow_raise_r",
    "au_brow_inner_dist",
]


def make_backbone(arch: str, pretrained: bool):
    arch = arch.lower()
    if arch == "resnet18":
        w = models.ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=w)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
        return m, feat_dim
    if arch == "resnet50":
        w = models.ResNet50_Weights.DEFAULT if pretrained else None
        m = models.resnet50(weights=w)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
        return m, feat_dim
    if arch == "mobilenet_v3_small":
        w = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        m = models.mobilenet_v3_small(weights=w)
        feat_dim = m.classifier[-1].in_features
        m.classifier[-1] = nn.Identity()
        return m, feat_dim
    if arch == "efficientnet_b0":
        w = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b0(weights=w)
        feat_dim = m.classifier[-1].in_features
        m.classifier[-1] = nn.Identity()
        return m, feat_dim
    if arch == "efficientnet_b2":
        w = models.EfficientNet_B2_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b2(weights=w)
        feat_dim = m.classifier[-1].in_features
        m.classifier[-1] = nn.Identity()
        return m, feat_dim
    if arch == "swin_tiny":
        import timm
        m = timm.create_model("swin_tiny_patch4_window7_224",
                              pretrained=pretrained, num_classes=0)
        feat_dim = m.num_features
        return m, feat_dim
    raise ValueError(f"Unsupported arch: {arch}")


class FERDataset(Dataset):
    def __init__(self, df, img_size, train):
        self.df       = df.reset_index(drop=True)
        self.img_size = img_size
        self.train    = train
        imagenet_mean = [0.485, 0.456, 0.406]
        imagenet_std  = [0.229, 0.224, 0.225]
        if train:
            self.tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
                transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.03),
                transforms.RandomGrayscale(p=0.05),
                transforms.ToTensor(),
                transforms.Normalize(imagenet_mean, imagenet_std),
                transforms.RandomErasing(p=0.10, scale=(0.02, 0.10)),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(imagenet_mean, imagenet_std),
            ])
        self.rare_aug = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.05),
            transforms.RandomAffine(degrees=10, translate=(0.10, 0.10), scale=(0.88, 1.12)),
            transforms.RandomPerspective(distortion_scale=0.15, p=0.4),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r        = self.df.iloc[idx]
        img_path = str(r["image"])
        label    = str(r["label"]).lower()
        y        = LAB2ID[label]
        soft = None
        if all(c in self.df.columns for c in P_COLS):
            vals = [r.get(c, np.nan) for c in P_COLS]
            if all(pd.notna(v) for v in vals):
                v = np.array(vals, dtype=np.float32)
                s = float(v.sum())
                if s > 1e-6:
                    soft = v / s
        if soft is None:
            soft = np.zeros((7,), dtype=np.float32)
            soft[y] = 1.0
        au_ok = int(r.get("au_ok", 0)) if "au_ok" in self.df.columns else 0
        au    = np.zeros((len(AU_COLS),), dtype=np.float32)
        if au_ok and all(c in self.df.columns for c in AU_COLS):
            vals = [r.get(c, np.nan) for c in AU_COLS]
            if all(pd.notna(v) for v in vals):
                au = np.array(vals, dtype=np.float32)
            else:
                au_ok = 0
        im = Image.open(img_path).convert("RGB")
        if self.train and label in RARE_CLASSES:
            im = self.rare_aug(im)
        x = self.tf(im)
        dataset_name = str(r.get("dataset", "unk"))
        return x, y, soft, au, au_ok, dataset_name


class FERModel(nn.Module):
    def __init__(self, arch, pretrained, num_classes=7, au_dim=9, use_au=True, dropout=0.2):
        super().__init__()
        self.backbone, feat_dim = make_backbone(arch, pretrained)
        self.dropout  = nn.Dropout(p=dropout)
        self.cls_head = nn.Linear(feat_dim, num_classes)
        self.use_au   = use_au
        if use_au:
            self.au_head = nn.Sequential(
                nn.Linear(feat_dim, 256),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.1),
                nn.Linear(256, au_dim),
            )

    def forward(self, x):
        feats   = self.backbone(x)
        feats_d = self.dropout(feats)
        logits  = self.cls_head(feats_d)
        au_pred = self.au_head(feats) if self.use_au else None
        return logits, au_pred


def soft_ce_loss(logits, target_probs, smoothing=0.05, weight=None):
    n_cls  = logits.size(1)
    smooth = target_probs * (1.0 - smoothing) + smoothing / n_cls
    logp   = F.log_softmax(logits, dim=1)
    loss   = -(smooth * logp).sum(dim=1)
    if weight is not None:
        hard_class = target_probs.argmax(dim=1)
        w = weight[hard_class]
        loss = loss * w
    return loss.mean()


def mixup_batch(x, soft, alpha=0.2):
    lam   = float(np.random.beta(alpha, alpha))
    idx   = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1.0 - lam) * x[idx]
    s_mix = lam * soft + (1.0 - lam) * soft[idx]
    return x_mix, s_mix


@torch.no_grad()
def eval_epoch(model, loader, device, smoothing=0.05, weight=None):
    model.eval()
    ys, yp   = [], []
    loss_sum = 0.0
    n        = 0
    for x, y, soft, au, au_ok, _ds in loader:
        x    = x.to(device)
        y    = y.to(device)
        soft = soft.to(device)
        logits, _ = model(x)
        loss = soft_ce_loss(logits, soft, smoothing=smoothing, weight=weight)
        loss_sum += float(loss.item()) * len(x)
        n        += len(x)
        pred = torch.argmax(logits, dim=1)
        ys.extend(y.cpu().numpy().tolist())
        yp.extend(pred.cpu().numpy().tolist())
    acc      = float((np.array(ys) == np.array(yp)).mean()) if ys else 0.0
    macro_f1 = float(f1_score(ys, yp, average="macro", zero_division=0)) if ys else 0.0
    return loss_sum / max(1, n), acc, macro_f1, ys, yp


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv",          required=True)
    ap.add_argument("--arch",         default="efficientnet_b0",
                    choices=["resnet18", "resnet50", "mobilenet_v3_small",
                             "efficientnet_b0", "efficientnet_b2", "swin_tiny"])
    ap.add_argument("--pretrained",   action="store_true")
    ap.add_argument("--img_size",     type=int,   default=160)
    ap.add_argument("--batch",        type=int,   default=32)
    ap.add_argument("--epochs",       type=int,   default=50)
    ap.add_argument("--lr",           type=float, default=3e-4)
    ap.add_argument("--weight_decay", type=float, default=1e-4)
    ap.add_argument("--num_workers",  type=int,   default=2)
    ap.add_argument("--seed",         type=int,   default=13)
    ap.add_argument("--balance",      default="dataset_class",
                    choices=["none", "dataset", "dataset_class"])
    ap.add_argument("--lambda_au",    type=float, default=0.05)
    ap.add_argument("--smoothing",    type=float, default=0.05)
    ap.add_argument("--dropout",      type=float, default=0.20)
    ap.add_argument("--freeze_epochs", type=int,  default=5)
    ap.add_argument("--mixup_alpha",  type=float, default=0.0)
    ap.add_argument("--save_dir",     default="models_cnn_multi")
    ap.add_argument("--run_name",     default="fer7_v4")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    df = pd.read_csv(args.csv, low_memory=False)
    for col in ["image", "label", "split", "dataset"]:
        if col not in df.columns:
            if col == "dataset":
                df["dataset"] = "unk"
            else:
                raise RuntimeError(f"CSV must contain column: {col}")
    df["image"]   = df["image"].astype(str)
    df["label"]   = df["label"].astype(str).str.lower()
    df["split"]   = df["split"].astype(str).str.lower()
    df["dataset"] = df["dataset"].astype(str).str.lower()
    for c in P_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in AU_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "au_ok" in df.columns:
        df["au_ok"] = pd.to_numeric(df["au_ok"], errors="coerce").fillna(0).astype(int)
    df = df[df["label"].isin(FER7)].reset_index(drop=True)
    df = df[df["image"].apply(lambda p: os.path.isfile(str(p)))].reset_index(drop=True)

    train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
    val_df   = df[df["split"] == "val"].copy().reset_index(drop=True)
    test_df  = df[df["split"] == "test"].copy().reset_index(drop=True)
    print(f"[DATA] train={len(train_df)} val={len(val_df)} test={len(test_df)}")
    print(f"[DATA] train class distribution:\n{train_df['label'].value_counts().to_string()}")

    use_au = (args.lambda_au > 0) and ("au_ok" in df.columns) and all(c in df.columns for c in AU_COLS)
    print(f"[AU]     use_au={use_au}  lambda_au={args.lambda_au}")
    print(f"[CONFIG] smoothing={args.smoothing}  dropout={args.dropout}  "
          f"freeze_epochs={args.freeze_epochs}  mixup_alpha={args.mixup_alpha}")

    label_counts  = train_df["label"].value_counts()
    raw_weights   = torch.tensor(
        [1.0 / label_counts.get(c, 1) for c in FER7], dtype=torch.float32
    )
    class_weights = (raw_weights / raw_weights.mean())
    print("[CLASS WEIGHTS]")
    for c, w in zip(FER7, class_weights.tolist()):
        print(f"  {c:10s}: {w:.3f}  (n={label_counts.get(c, 0)})")

    ds_train = FERDataset(train_df, img_size=args.img_size, train=True)
    ds_val   = FERDataset(val_df,   img_size=args.img_size, train=False) if len(val_df)  else None
    ds_test  = FERDataset(test_df,  img_size=args.img_size, train=False) if len(test_df) else None

    sampler = None
    if args.balance != "none":
        if args.balance == "dataset":
            ds_counts  = train_df["dataset"].value_counts().to_dict()
            weights_np = train_df["dataset"].map(
                lambda d: 1.0 / ds_counts.get(d, 1)
            ).to_numpy(dtype=np.float64)
        elif args.balance == "dataset_class":
            tmp        = train_df.copy()
            tmp["key"] = tmp["dataset"].astype(str) + "::" + tmp["label"].astype(str)
            key_counts = tmp["key"].value_counts().to_dict()
            weights_np = tmp["key"].map(
                lambda k: 1.0 / key_counts.get(k, 1)
            ).to_numpy(dtype=np.float64)
        else:
            weights_np = np.ones(len(train_df), dtype=np.float64)
        weights = torch.as_tensor(weights_np, dtype=torch.double)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        print(f"[BALANCE] {args.balance} enabled")

    pin      = torch.cuda.is_available()
    pw       = args.num_workers > 0
    dl_train = DataLoader(ds_train, batch_size=args.batch, sampler=sampler,
                          shuffle=(sampler is None), num_workers=args.num_workers,
                          pin_memory=pin, persistent_workers=pw)
    dl_val   = DataLoader(ds_val,  batch_size=args.batch, shuffle=False,
                          num_workers=args.num_workers, pin_memory=pin,
                          persistent_workers=pw) if ds_val  else None
    dl_test  = DataLoader(ds_test, batch_size=args.batch, shuffle=False,
                          num_workers=args.num_workers, pin_memory=pin,
                          persistent_workers=pw) if ds_test else None

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[DEVICE] {device}")

    model = FERModel(
        args.arch, args.pretrained,
        num_classes=7, au_dim=len(AU_COLS),
        use_au=use_au, dropout=args.dropout
    ).to(device)

    cw = class_weights.to(device)

    backbone_frozen = args.freeze_epochs > 0 and args.pretrained
    if backbone_frozen:
        for p in model.backbone.parameters():
            p.requires_grad = False
        print(f"[FREEZE] backbone frozen for first {args.freeze_epochs} epochs")

    def build_optimizer(frozen):
        if frozen:
            return torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=args.lr, weight_decay=args.weight_decay
            )
        else:
            param_groups = [
                {"params": model.backbone.parameters(), "lr": args.lr * 0.1},
                {"params": model.cls_head.parameters(), "lr": args.lr},
            ]
            if use_au:
                param_groups.append({"params": model.au_head.parameters(), "lr": args.lr})
            return torch.optim.AdamW(param_groups, weight_decay=args.weight_decay)

    def build_scheduler(opt, n_epochs):
        return torch.optim.lr_scheduler.OneCycleLR(
            opt,
            max_lr=[g["lr"] for g in opt.param_groups],
            steps_per_epoch=len(dl_train),
            epochs=n_epochs,
            pct_start=0.1,
            anneal_strategy="cos",
            div_factor=10.0,
            final_div_factor=1e4,
        )

    opt   = build_optimizer(backbone_frozen)
    sched = build_scheduler(opt, args.epochs)
    au_loss_fn = nn.SmoothL1Loss(reduction="mean")

    save_dir  = Path(args.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    best_path = save_dir / f"{args.run_name}_best.pt"
    meta_path = save_dir / f"{args.run_name}_meta.json"
    best_f1 = -1.0

    for ep in range(1, args.epochs + 1):
        if backbone_frozen and ep == args.freeze_epochs + 1:
            for p in model.backbone.parameters():
                p.requires_grad = True
            backbone_frozen = False
            remaining       = args.epochs - ep + 1
            opt             = build_optimizer(frozen=False)
            sched           = build_scheduler(opt, remaining)
            print(f"[UNFREEZE] backbone unfrozen at epoch {ep}, "
                  f"new max_lr={args.lr * 0.1:.2e} (backbone) / {args.lr:.2e} (head) "
                  f"over {remaining} remaining epochs")

        model.train()
        loss_sum = 0.0
        n        = 0
        for x, y, soft, au, au_ok, _ds in dl_train:
            x     = x.to(device)
            soft  = soft.to(device)
            au    = au.to(device)
            au_ok = au_ok.to(device)
            if args.mixup_alpha > 0:
                x, soft = mixup_batch(x, soft, alpha=args.mixup_alpha)
            logits, au_pred = model(x)
            loss_cls = soft_ce_loss(logits, soft, smoothing=args.smoothing, weight=cw)
            loss = loss_cls
            if use_au:
                mask = (au_ok > 0)
                if mask.any():
                    loss_au = au_loss_fn(au_pred[mask], au[mask])
                    loss    = loss + args.lambda_au * loss_au
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            sched.step()
            loss_sum += float(loss.item()) * len(x)
            n        += len(x)

        train_loss = loss_sum / max(1, n)
        current_lr = sched.get_last_lr()[0] if hasattr(sched, "get_last_lr") else args.lr

        if dl_val is not None:
            val_loss, val_acc, val_f1, ys, yp = eval_epoch(
                model, dl_val, device, smoothing=args.smoothing, weight=cw
            )
        else:
            val_loss, val_acc, val_f1 = 0.0, 0.0, 0.0

        frozen_tag = " [frozen]" if backbone_frozen else ""
        print(
            f"[EPOCH {ep:03d}/{args.epochs}] "
            f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"val_acc={val_acc:.3f}  val_macro_f1={val_f1:.3f}  "
            f"lr={current_lr:.2e}{frozen_tag}"
        )

        if dl_val is not None and val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                "state_dict": model.state_dict(),
                "arch":       args.arch,
                "img_size":   args.img_size,
                "epoch":      ep,
                "val_f1":     val_f1,
                "val_acc":    val_acc,
            }, best_path)
            print(f"  [SAVE] best val_macro_f1={val_f1:.3f} -> {best_path}")

    if best_path.exists():
        ckpt = torch.load(best_path, map_location=device)
        model.load_state_dict(ckpt["state_dict"])
        print(f"\n[BEST CKPT] epoch={ckpt.get('epoch','?')}  "
              f"val_f1={ckpt.get('val_f1', 0):.3f}")

    if dl_test is not None:
        test_loss, test_acc, test_f1, ys, yp = eval_epoch(
            model, dl_test, device, smoothing=0.0, weight=None
        )
        print(f"\n[TEST] acc={test_acc:.3f}  macro_f1={test_f1:.3f}")
        print("[TEST] report:\n",
              classification_report(ys, yp, target_names=FER7, digits=3))

    meta = {
        "classes":       FER7,
        "arch":          args.arch,
        "img_size":      args.img_size,
        "pretrained":    bool(args.pretrained),
        "use_au":        bool(use_au),
        "au_cols":       AU_COLS if use_au else [],
        "lambda_au":     args.lambda_au,
        "smoothing":     args.smoothing,
        "dropout":       args.dropout,
        "freeze_epochs": args.freeze_epochs,
        "mixup_alpha":   args.mixup_alpha,
        "csv":           str(Path(args.csv).resolve()),
        "best_ckpt":     str(best_path.resolve()),
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print("[OK] wrote", meta_path)


if __name__ == "__main__":
    main()

## 6. Run Training!
This should take **1-3 hours** on a Colab GPU (vs days on CPU).

In [ ]:
!python /content/cnn_train_multidataset_fer7_soft_au.py \
  --csv /content/data/unified_fer7_processed_au.csv \
  --arch efficientnet_b0 \
  --pretrained \
  --img_size 224 \
  --lr 1e-4 \
  --epochs 50 \
  --mixup_alpha 0.2 \
  --batch 32 \
  --num_workers 2 \
  --save_dir /content/drive/MyDrive/FER/models \
  --run_name fer7_v4_colab

## 7. Copy best model back to Drive
The model is already saved to Drive (via `--save_dir`). You can download it from Google Drive to your local PC for webcam inference.

In [ ]:
import os

MODELS_DIR = '/content/drive/MyDrive/FER/models'
print("Saved models:")
for f in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

print("\nDownload these to your local models_cnn_multi/ folder:")
print("  - fer7_v4_colab_best.pt")
print("  - fer7_v4_colab_meta.json")
print("Then run: python cnn_webcam_fer7_multi.py --run_name fer7_v4_colab")